# Geant4 Simulation

The Geant4 simulation step is essential for accurately modeling the passage of particles through matter. It simulates the interactions of particles with the detector materials, allowing us to understand how particles lose energy, scatter, and produce secondary particles. This step is crucial for obtaining realistic hit information that will be used in subsequent analysis stages.

To run the Geant4 simulation, we typically use the `simu_trf.py` script, which takes the output from the event generation step as input. The script can be configured to enable various features, such as magnetic fields, which are important for simulating charged particle trajectories accurately.

In [1]:
!simu_trf.py

Module libc not found.
Usage: simu_trf.py -i INPUT_FILE -o OUTPUT_FILE [--nov NUMBER_OF_EVENTS]
                   [-nt NUMBER_OF_THREADS] [--enable-magnetic-field]
                   [-t TIMEOUT] [-l OUTPUT_LEVEL] [--pre-init PRE_INIT]
                   [--pre-exec PRE_EXEC] [--post-exec POST_EXEC]
                   [--save-all-hits] [--dry-run] [--job-file JOB_FILE]

Options:
  -i, --input-file INPUT_FILE
                        The input file or folder to run the job
  -o, --output-file OUTPUT_FILE
                        The output file.
  --nov, --number-of-events NUMBER_OF_EVENTS
                        The total number of events to run.
  -nt, --number-of-threads NUMBER_OF_THREADS
                        The number of threads
  --enable-magnetic-field
                        Enable the magnetic field.
  -t, --timeout TIMEOUT
                        Event timeout in minutes
  -l, --output-level OUTPUT_LEVEL
                        The output level messenger.
  --pre-init PRE_IN

In [2]:
!simu_trf.py -i Zee.EVT.root -o Zee.HIT.root --nov 10 -nt 10 --dry-run

Module libc not found.
input file: Zee.EVT.root
output file: Zee.HIT.root
number of threads: 10
Cxx.ComponentAccumulator                 INFO Run manager was created.
Compiling...: 100%|█████████████████| 34/34 [00:00<00:00, 3267.04it/s]
Py.CaloHitBuilder                       INFO Configure CaloHitBuilder.
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_PSB collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB2 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB3 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_TileCal1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into C

## Magnetic field

By default the simulation runs without magnetic field. Two options enable a 2 T axial field:

* `--enable-solenoid-field`: the field exists only inside the volume of the ATLAS central solenoid, a cylinder with radius 1.23 m and half-length 2.9 m (JINST 3 (2008) S08003, table 2.1). Particles are bent on their way to the calorimeters, and the showers inside the calorimeters are not. The solenoid coil and cryostat material are not modelled, and the field is uniform (in ATLAS it drops from about 1.8 T at $z$ = 1.7 m to about 0.9 T at the end of the inner detector cavity).
* `--enable-magnetic-field`: a uniform 2 T field in the whole world, including the calorimeters.

The two options cannot be used together. With `--enable-solenoid-field`, soft charged particles curl up inside the solenoid instead of being removed at generation, so the events should be generated without the pT cut, as ATLAS does:

```
gen_minbias.py ... --pt-min-charged 0 --pt-min-neutral 0
simu_trf.py -i MB.EVT.root -o MB.HIT.root --enable-solenoid-field
```

## Pre-execution adaptation:

It's possible to configure certain algorithms within the simulation using the `pre-exec` parameter. Usually this stage can be used when all algorithms are setted. This will be executed before the execution (`run`) event loop.

### Save all hits:

Here is hot can set the simulator to keep all hits.

In [3]:
!simu_trf.py -i Zee.EVT.root -o Zee.HIT.root --nov 10 -nt 10 --dry-run --pre-exec "HIT.setProperty('OnlyRoI',False)"

Module libc not found.
input file: Zee.EVT.root
output file: Zee.HIT.root
number of threads: 10
Cxx.ComponentAccumulator                 INFO Run manager was created.
Compiling...: 100%|█████████████████| 34/34 [00:00<00:00, 3428.94it/s]
Py.CaloHitBuilder                       INFO Configure CaloHitBuilder.
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_PSB collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB2 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB3 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_TileCal1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into C

### Anomaly simulation:

Here is how you can set the KeepCells parameter for the `RootStreamHitMaker` algorithm:
* Parameter: `KeepCells`
* Value: A list of cells specified by their hash number.

**NOTE**: This specific strategy is often utilized in anomaly simulations.

In [4]:
import json

cells = [
                        # examples of cell hash IDs where noise is increased
                             1265,      1266,      1267,      1268,      1269, # center PSB
                        170001265, 170001266, 170001267, 170001268, 170001269, # right PSB
                        340001265, 340001266, 340001267, 340001268, 340001269, # left PSB
                         18400805,  18400806,  18400807,  18400808,  18400809, # center PSE
                        188400805, 188400806, 188400807, 188400808, 188400809, # right PSE
                        358000805, 358000806, 358000807, 358000808, 358000809, # left PSE
                         20025610,  20025611,  20025612,  20025613,  20025614, # center EMB1
                        190025610, 190025611, 190025612, 190025613, 190025614, # right EMB1
                        360025610, 360025611, 360025612, 360025613, 360025614, # left EMB1
                         30029160,  30029161,  30029162,  30029163,  30029164, # center EMB2
                        200029160, 200029161, 200029162, 200029163, 200029164, # right EMB2
                        370029160, 370029161, 370029162, 370029163, 370029164, # left EMB2
                         40013020,  40013021,  40013022,  40013023,  40013024, # center EMB3
                        210013020, 210013021, 210013022, 210013023, 210013024, # right EMB3
                        380013020, 380013021, 380013022, 380013023, 380013024, # left EMB3
                         50000245,  50000246,  50000247,  50000248,  50000249, # center TileCal1
                        220000245, 220000246, 220000247, 220000248, 220000249, # right TileCal1
                        390000245, 390000246, 390000247, 390000248, 390000249, # left TileCal1
                         60000245,  60000246,  60000247,  60000248,  60000249, # center TileCal2
                        230000245, 230000246, 230000247, 230000248, 230000249, # right TileCal2
                        400000245, 400000246, 400000247, 400000248, 400000249, # left TileCal2
                         78406245,  78406246,  78406247,  78406248,  78406249, # center TileCal3
                        248406245, 248406246, 248406247, 248406248, 248406249, # right TileCal3
                        418406245, 418406246, 418406247, 418406248, 418406249, # left TileCal3
                        256000680, 256000681, 256000682, 256000683, 256000684, # right TileExt1
                        426000680, 426000681, 426000682, 426000683, 426000684, # left TileExt1
                        260000120, 260000121, 260000122, 260000123, 260000124, # right TileExt2
                        430000120, 430000121, 430000122, 430000123, 430000124, # left TileExt2
                        270000106, 270000107, 270000108, 270000109, 270000110, # right TileExt3
                        440000106, 440000107, 440000108, 440000109, 440000110, # left TileExt3
                        280000440, 280000441, 280000442, 280000443, 280000444, # right EMEC1
                        450000440, 450000441, 450000442, 450000443, 450000444, # left EMEC1
                        290000440, 290000441, 290000442, 290000443, 290000444, # right EMEC2
                        460000440, 460000441, 460000442, 460000443, 460000444, # left EMEC2
                        300005110, 300005111, 300005112, 300005113, 300005114, # right EMEC3
                        470005110, 470005111, 470005112, 470005113, 470005114, # left EMEC3
                        312807780, 312807781, 312807782, 312807783, 312807784, # right HEC1
                        482807780, 482807781, 482807782, 482807783, 482807784, # left HEC1                        
                        320000560, 320000561, 320000562, 320000563, 320000564, # right HEC2
                        490000560, 490000561, 490000562, 490000563, 490000564, # left HEC2
                        330000290, 330000291, 330000292, 330000293, 330000294, # right HEC3
                        500000290, 500000291, 500000292, 500000293, 500000294, # left HEC3
      ]

d = {
    "runs": [
        {
            "DeadModules"     : True,
            "StartEventNumber": 80000,
            "EndEventNumber"  : 81000,
            "NoiseStdFactor"  : 2,
            "Cells"           : [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
        },
        {
            "DeadModules"     : True,
            "StartEventNumber": 85000,
            "EndEventNumber"  : 87000,
            "NoiseStdFactor"  : 2,
            "Cells"           : [40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
        },
        {
            "DeadModules"     : False,
            "StartEventNumber": 89000,
            "EndEventNumber"  : 92000,
            "NoiseStdFactor"  : 2,
            "Cells"           : cells,    
        }
    ] 
}
with open("brl.json", 'w') as f:
    json.dump(d,f)



In [5]:
!simu_trf.py -i Zee.EVT.root -o Zee.HIT.root --nov 1 --pre-exec "from CaloCellBuilder import get_cells_from_brl as reader;HIT.setProperty('KeepCells',reader('brl.json'))"

Module libc not found.
input file: Zee.EVT.root
output file: Zee.HIT.root
number of threads: 1
Cxx.ComponentAccumulator                 INFO Run manager was created.
Compiling...: 100%|█████████████████| 34/34 [00:00<00:00, 3278.76it/s]
Py.CaloHitBuilder                       INFO Configure CaloHitBuilder.
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_PSB collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB2 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_EMB3 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Collection_TileCal1 collection
Py.CaloHitBuilder                       INFO Create new CaloHitMaker and dump all hits into Co

In [6]:
!ls

Simulation.ipynb  Zee.ESD.root	Zee.HIT.root  dead_modules.json
Untitled.ipynb	  Zee.EVT.root	brl.json
